# tp-09 馬券戦略の比較（文献に基づく手法の実装比較）

**実行場所: ホスト PC（stg）。** tp-08 は「確定オッズに対して控除率を超える優位は見つからず、回収率は最良でも約 88%」だった。文献調査（Benter 1994、Kelly 系 = Uhrín ほか 2021・Busseti ほか 2016・Sun & Boyd 2018、Bjerksund & Stensland 2017、optimizer's curse、確定オッズ先読みの指摘 = 日本語の検証記事ほか）に沿って、
**優位が本当に無いのか**を厳密に確かめ、**学習のしかたを変えると損失がどこまで縮むか**を実装して比べる。

## 内容
1. **優位の有無の検定**: (a) Benter ブレンドの a（モデルの重み）= 0 の尤度比検定と bootstrap 区間、(b) 対照実験（モデルの予測をレース内でシャッフル）、(c) 期待回収率に対する実現回収率の傾き（1 = 較正済み、0 以下 = 優位なし）。
2. **着順分布の指数の最尤推定**: 市場の単勝確率だけから作る割引 Harville の α2・α3 を、実際の 1〜3 着の並びで最尤推定（レースごとの当てはめとの差を見る）。
3. **券種間の整合（Bjerksund–Stensland 型）**: 単勝の暗黙確率から作った馬連・ワイド・馬単の「公正な配当」と、実際の配当の比（cross 値 = p_mkt × オッズ）の十分位別の実現回収率。
4. **戦略の比較**（valid 2023・2024）:
   - RL 基本（tp-08 の最良設定）／ RL + cross 特徴 ／ RL + cross + 較正した期待回収率（cal）／ RL の目的を資金の対数成長（Kelly）に替えたもの
   - 選別の基準: 較正した期待回収率（等張回帰。optimizer's curse の縮小）の上位 k 点、cross 値の上位 k 点（参加率 100% / 50%）
   - 参考: 複勝・馬連のモデル 1 位、ランダム
   - **購入時点のオッズのずれの模擬**: 判断に使うオッズに対数正規の雑音（σ = 0, 0.1）を入れ、払戻は確定値で精算。雑音を入れた学習・評価と、雑音なしで学習した方策に雑音つきオッズを与えた頑健性も見る。
5. **test 2025（一度だけ）**: 上記すべてを事前に決めた形で評価（比較研究なので選んだものだけの報告にはしない）。

## 注意
- オッズは締切後の確定オッズ（購入時点の値ではない）。雑音の模擬は粗い代理で、実際のずれは T-60/T-15 のスナップショットが貯まってから測る。
- 方策の賭け金の下限（平均 0.5）は tp-08 と同じ。下限が無いと全見送りに落ちる（優位が無いため）。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "4")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, time, gc
from datetime import datetime
import numpy as np
import pandas as pd
import mlflow

from src.pipeline.betting import market_analysis as ma
from src.pipeline.betting import policy as pol
from src.pipeline.betting import ticket_env as te
from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models.compute import detect_compute

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
BET = REPO_ROOT / "data/local/modeling/betting"
TAG = os.environ.get("TP09_TAG", "cmp_v2")
OOF_FILE = Path(os.environ.get("TP09_OOF", WORK / "oof/model_oof/ensemble_v2.parquet"))
TEST_FILE = Path(os.environ.get("TP09_TEST", WORK / "final_eval_v2/test2025_predictions.parquet"))
SMOKE = os.environ.get("TP09_SMOKE") == "1"          # 動作確認用: test 2025 を使わず、2022〜2023 で学習・2024 で評価。seed と試行を減らす
K = 12
OUT = BET / "rl_compare" / TAG
OUT.mkdir(parents=True, exist_ok=True)
LOCK = BET / "rl_compare" / f"TEST2025_EVALUATED_{TAG}.json"
THREADS = max(1, min(4, detect_compute(reserve_cores=6).threads))
SEEDS = (0,) if SMOKE else (0, 1, 2)
FINAL_SEEDS = (0, 1) if SMOKE else (0, 1, 2, 3, 4)
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
mlflow.set_experiment("keiba-tp-betting-compare")
require_tracking_server()
YEARS = ["2022", "2023", "2024", "2025"]
wp = pd.read_parquet(BET / "odds_win_place.parquet"); wp = wp[wp["race_id"].str[:4].isin(YEARS) & (wp["win_odds"] > 0)].reset_index(drop=True)       # win_odds = -3.0 は取消・除外（出走しない）
pair = pd.read_parquet(BET / "odds_pair.parquet"); pair = pair[pair["race_id"].str[:4].isin(YEARS)].reset_index(drop=True)
pay = pd.read_parquet(BET / "payoff.parquet"); pay = pay[pay["race_id"].str[:4].isin(YEARS)].reset_index(drop=True)
cols = ["race_id", "horse_id", "year", "p_win", "p_top2", "p_top3"]
pred_all = pd.concat([pd.read_parquet(OOF_FILE)[cols], pd.read_parquet(TEST_FILE)[cols]], ignore_index=True)
pred_all = pred_all[pred_all["year"].between(2022, 2025)]
BEST8 = json.loads((BET / "rl/ens_v2/best_config.json").read_text(encoding="utf-8"))["params"]
print(f"予測 {len(pred_all):,} 行 / スレッド {THREADS} / seeds {SEEDS} / SMOKE={SMOKE}")
print("tp-08 の最良設定:", BEST8)

## 1. 優位の有無の検定
Benter の (a, b) は 2022 年だけで当てはめる（tp-08 と同じ）。a = 0 の尤度比検定と、レース単位 bootstrap の区間。対照として、モデルの予測をレース内でシャッフルしたもの（情報ゼロ）を同じ手順で当てはめる。

In [ ]:
win_no = pay[pay["bet_type"] == "tansho"].groupby("race_id")["combo"].agg(["count", "first"])
win_no = win_no[win_no["count"] == 1]["first"].astype(int)
m = pred_all.merge(wp[["race_id", "horse_id", "horse_number", "win_odds"]], on=["race_id", "horse_id"], how="inner").dropna(subset=["win_odds"])
m = m[m["race_id"].isin(win_no.index)]
m["won"] = m["horse_number"].to_numpy() == win_no.reindex(m["race_id"]).to_numpy()
GROUPS = {}
for rid, g in m.groupby("race_id", sort=False):
    if g["won"].sum() != 1:
        continue
    pm = g["p_win"].to_numpy(); pm = pm / pm.sum()
    GROUPS[rid] = (np.log(np.clip(pm, 1e-12, None)), np.log(te.market_probs(g["win_odds"].to_numpy())), int(np.flatnonzero(g["won"].to_numpy())[0]))
by_year = {y: [v for r, v in GROUPS.items() if r[:4] == y] for y in YEARS}
rng = np.random.default_rng(0)
rows = []
for y in ["2022", "2023", "2024"]:
    t = ma.lr_test_model_weight(by_year[y], n_boot=60 if SMOKE else 200)
    rows.append({"年": y, "対象": "モデル", **t})
shuf = [(rng.permutation(lm), lk, w) for lm, lk, w in by_year["2022"]]
rows.append({"年": "2022", "対象": "対照（モデルをレース内でシャッフル）", **ma.lr_test_model_weight(shuf, n_boot=60 if SMOKE else 200)})
LR = pd.DataFrame(rows).set_index(["年", "対象"])
display(LR[["a", "a_lo", "a_hi", "b", "b_lo", "b_hi", "gain_per_race", "lr", "p_value"]].round(4))
LR.to_csv(OUT / "lr_test.csv")
A_B, B_B = te.fit_benter(by_year["2022"])
print(f"tp-08 と同じ (a, b) = ({A_B:.3f}, {B_B:.3f})")

## 2. 着順分布の指数の最尤推定（市場の単勝確率のみ）
2022 年の 1〜3 着の並び（3連単の払戻から復元）で α2・α3 を最尤推定し、2023・2024 年の平均負対数尤度で、Harville（α=1）と比べる。

In [ ]:
orders = ma.finishing_orders(pay)
wpm = {rid: (g["horse_number"].to_numpy(), te.market_probs(g["win_odds"].to_numpy())) for rid, g in wp[wp["win_odds"].notna()].groupby("race_id")}
G = {y: ma.order_groups({r: v for r, v in wpm.items() if r[:4] == y}, orders) for y in YEARS[:3]}
fit_g = G["2022"][:1200] if SMOKE else G["2022"][:2000]
t0 = time.time()
AL2, AL3, nll_fit = ma.fit_global_alphas(fit_g)
rows = [{"設定": "Harville (α2=α3=1)", "α2": 1.0, "α3": 1.0}, {"設定": "最尤推定（2022 年）", "α2": AL2, "α3": AL3}]
for r in rows:
    for y in ("2023", "2024"):
        r[f"負対数尤度 {y}"] = ma.order_nll(G[y][:1500], r["α2"], r["α3"])
AL = pd.DataFrame(rows).set_index("設定")
display(AL.round(4)); AL.to_csv(OUT / "alpha_mle.csv")
print(f"最尤推定 {time.time() - t0:.0f}s / 並びが復元できたレース {({y: len(G[y]) for y in G})}")

## 3. 買い目の表（市場だけの確率 p_mkt つき）
tp-08 の表に、市場の単勝確率だけから（最尤推定の α で）作った組の確率 `p_mkt` を足す。候補には「p_mkt × オッズ」が高い順（券種間の整合ずれ）も加える。

In [ ]:
TDIR = BET / "tickets" / f"{TAG}_K{K}_v2"
TDIR.mkdir(parents=True, exist_ok=True)
TICK = {}
for y in YEARS:
    f, mf = TDIR / f"{y}.parquet", TDIR / f"{y}.json"
    want = {"a": round(A_B, 6), "b": round(B_B, 6), "K": K, "al": [round(AL2, 4), round(AL3, 4)], "clean": 1}
    if f.exists() and mf.exists() and json.loads(mf.read_text()) == want:
        TICK[y] = pd.read_parquet(f); print(y, "キャッシュ", len(TICK[y])); continue
    t0 = time.time()
    ids = set(pred_all.loc[pred_all["year"] == int(y), "race_id"])
    d = te.build_parallel(pred_all[pred_all["race_id"].isin(ids)], wp[wp["race_id"].isin(ids)], pair[pair["race_id"].isin(ids)], pay[pay["race_id"].isin(ids)], A_B, B_B, K=K, procs=THREADS, chunks=40, mkt_alphas=(AL2, AL3))
    d.to_parquet(f, index=False); mf.write_text(json.dumps(want)); TICK[y] = d
    print(f"{y}: {d['race_id'].nunique():,} レース、{len(d):,} 点 ({time.time() - t0:.0f}s)")

## 4. 期待回収率の較正と券種間の整合（valid 2023・2024）
- **傾き**: 実現損益（mult − 1）を期待損益（p × オッズ − 1）に回帰した傾き。1 なら較正済み、0 以下なら「期待値が高いほど儲かる」関係が無い。レース単位 bootstrap の 95% 区間。
- **cross 値** = p_mkt × オッズ: 単勝の暗黙確率から作った公正な配当に対する、実際の配当の高さ。十分位別の実現回収率。

In [ ]:
VAL = pd.concat([TICK["2023"], TICK["2024"]], ignore_index=True)
rows = []
for name, pc in (("ブレンド確率 p_comb", "p_comb"), ("モデル単独 p_model", "p_model"), ("市場のみ p_mkt（cross 値）", "p_mkt")):
    for ty in ["all"] + te.TYPES:
        d = VAL if ty == "all" else VAL[VAL["type"] == ty]
        r = ma.ev_slope(d.assign(ev_hat=d[pc] * d["odds"]), n_boot=100 if SMOKE else 300)
        rows.append({"確率": name, "券種": ty, **r})
SL = pd.DataFrame(rows).set_index(["確率", "券種"])
display(SL.round(3)); SL.to_csv(OUT / "ev_slope.csv")
x = VAL["p_mkt"] * VAL["odds"]
dec = VAL.assign(x=x)
dec["decile"] = dec.groupby("type")["x"].transform(lambda s: pd.qcut(s.rank(method="first"), 10, labels=False))
CR = dec.groupby(["type", "decile"]).agg(n=("mult", "size"), cross値=("x", "mean"), 実現回収率=("mult", "mean")).round(3)
display(CR.unstack(0)["実現回収率"]); CR.to_csv(OUT / "cross_deciles.csv")

## 5. 戦略の比較（valid 2023・2024）
fold A = 学習 2022 → 評価 2023、fold B = 学習 2022〜2023 → 評価 2024。RL の設定は tp-08 の最良をそのまま使う（再調整なしの統制比較）。較正した期待回収率（cal）は、学習側は年内を半々に分けたクロスフィットで作る（学習データでの過適合を避ける）。

In [ ]:
def add_cal(dtr, dva):
    """較正した期待回収率 cal_ev（= 等張回帰した実現回収率）。学習側はレースの偶奇でクロスフィット、評価側は学習全体で当てはめ。"""
    xtr = np.clip(dtr["p_comb"].to_numpy() * dtr["odds"].to_numpy(), 0, 5)
    xva = np.clip(dva["p_comb"].to_numpy() * dva["odds"].to_numpy(), 0, 5)
    half = (pd.util.hash_array(dtr["race_id"].to_numpy()) % 2).astype(int)
    cal = np.zeros(len(dtr))
    for h in (0, 1):
        iso = ma.IsotonicEV().fit(dtr[half != h], xtr[half != h])
        cal[half == h] = iso.predict(dtr[half == h], xtr[half == h])
    dtr = dtr.assign(cal_ev=cal)
    dva = dva.assign(cal_ev=ma.IsotonicEV().fit(dtr, xtr).predict(dva, xva))
    return dtr, dva

CACHE = {}
def prepare(tr_years, te_year, sigma_tr=0.0, sigma_te=0.0, seed=0):
    key = (tuple(tr_years), te_year, sigma_tr, sigma_te)
    if key not in CACHE:
        dtr = pol.perturb_odds(pd.concat([TICK[y] for y in tr_years], ignore_index=True), sigma_tr, seed=11 + seed).reset_index(drop=True)
        dte = pol.perturb_odds(TICK[te_year], sigma_te, seed=22 + seed).reset_index(drop=True)
        CACHE[key] = add_cal(dtr, dte)
        if len(CACHE) > 6:
            CACHE.pop(next(iter(CACHE)))
    return CACHE[key]

RL_METHODS = {
    "RL 基本（tp-08）": dict(groups=("base",), mode="expected"),
    "RL + cross": dict(groups=("base", "cross"), mode="expected"),
    "RL + cross + cal": dict(groups=("base", "cross", "cal"), mode="expected"),
    "RL Kelly（対数成長）+ cross": dict(groups=("base", "cross"), mode="kelly"),
}
def run_rl(spec, tr_years, te_year, seeds, sigma_tr=0.0, sigma_te=0.0, cfg_over=None):
    dtr, dte = prepare(tr_years, te_year, sigma_tr, sigma_te)
    pk_tr, mu, sd = pol.pack(dtr, groups=spec["groups"])
    pk_te, _, _ = pol.pack(dte, mu, sd, groups=spec["groups"])
    ws = []
    for s in seeds:
        c = pol.TrainConfig(**{**BEST8, "mode": spec["mode"], "seed": s, "threads": THREADS, **(cfg_over or {})})
        net, _ = pol.train_policy(pk_tr, c)
        ws.append(pol.predict_alloc(net, pk_te, c.max_ticket))
    return pk_te, np.mean(ws, axis=0)

def select_methods(pk, dte):
    cal = pol.pack_column(pk, dte, "cal_ev")
    cross = pol.pack_column(pk, dte.assign(_x=dte["p_mkt"] * dte["odds"]), "_x")
    return {
        "較正EV上位3点（参加100%）": pol.alloc_topk_score(pk, cal, 3, 1.0),
        "較正EV上位3点（参加50%）": pol.alloc_topk_score(pk, cal, 3, 0.5),
        "cross値上位3点（参加100%）": pol.alloc_topk_score(pk, cross, 3, 1.0),
        "cross値上位3点（参加50%）": pol.alloc_topk_score(pk, cross, 3, 0.5),
        "複勝: モデル1位に全額": pol.alloc_top_p(pk, "fukusho"),
        "馬連: モデル1位に全額": pol.alloc_top_p(pk, "umaren"),
        "ランダム1点": pol.alloc_random(pk),
    }

FOLDS = [("A", ["2022"], "2023"), ("B", ["2022", "2023"], "2024")]
SIGMAS = [0.0, 0.1]
rows = []
for sigma in SIGMAS:
    for fold, tr_y, va_y in FOLDS:
        for name, spec in RL_METHODS.items():
            t0 = time.time()
            pk, w = run_rl(spec, tr_y, va_y, SEEDS, sigma, sigma)
            s = pol.summarize(pol.settle(pk, w), n_boot=300)
            rows.append({"雑音σ": sigma, "fold": fold, "方策": name, **{k: s[k] for k in ("roi", "roi_lo", "roi_hi", "profit_per_race", "stake_per_race", "participation")}, "sec": time.time() - t0})
            print(f"σ={sigma} {fold} {name:22s} ROI {s['roi']:.4f} 賭け金 {s['stake_per_race']:.3f} ({time.time() - t0:.0f}s)", flush=True)
        dtr, dte = prepare(tr_y, va_y, sigma, sigma)
        pk, _, _ = pol.pack(dte)
        for name, w in select_methods(pk, dte).items():
            s = pol.summarize(pol.settle(pk, w), n_boot=300)
            rows.append({"雑音σ": sigma, "fold": fold, "方策": name, **{k: s[k] for k in ("roi", "roi_lo", "roi_hi", "profit_per_race", "stake_per_race", "participation")}, "sec": 0.0})
        gc.collect()
CMP = pd.DataFrame(rows)
CMP.to_csv(OUT / "valid_compare.csv", index=False)
piv = CMP.pivot_table(index="方策", columns=["雑音σ", "fold"], values="roi")
piv["平均(σ=0)"] = piv[[(0.0, "A"), (0.0, "B")]].mean(axis=1); piv["平均(σ=0.1)"] = piv[[(0.1, "A"), (0.1, "B")]].mean(axis=1)
display(piv.sort_values("平均(σ=0)", ascending=False).round(4))

## 6. 最終: test 2025（一度だけ）
学習 2022〜2024。RL は seed 5 本の平均配分。選別系は 2022〜2024 で当てはめた較正を使う。購入時点のずれの模擬は、(i) 雑音なしで学習した方策に雑音つきオッズ（σ = 0.05, 0.1, 0.2）を与えた頑健性と、(ii) 雑音（σ = 0.1）つきで学習・評価した版の 2 通り。

In [ ]:
if not SMOKE and LOCK.exists() and os.environ.get("TP09_FORCE") != "1":
    raise SystemExit(f"test 2025 は評価済みです（{LOCK}）。結果を見てやり直さないため、再実行しません。")
TEST_Y = "2024" if SMOKE else "2025"
TR_Y = ["2022", "2023"] if SMOKE else ["2022", "2023", "2024"]
FIN = {}      # 名前 → (pk, 配分)
ref_pk = None
for name, spec in RL_METHODS.items():
    pk, w = run_rl(spec, TR_Y, TEST_Y, FINAL_SEEDS)
    FIN[name] = (pk, w); ref_pk = pk
    print("学習済み:", name, flush=True)
dtr, dte = prepare(TR_Y, TEST_Y)
pk_sel, _, _ = pol.pack(dte)
for name, w in select_methods(pk_sel, dte).items():
    FIN[name] = (pk_sel, w)
assert (ref_pk.race_ids == pk_sel.race_ids).all()
SET = {k: pol.settle(p, w) for k, (p, w) in FIN.items()}
rows = []
for k, s in SET.items():
    r = pol.summarize(s, n_boot=2000)
    rows.append({"方策": k, **{c: r[c] for c in ("roi", "roi_lo", "roi_hi", "profit_per_race", "stake_per_race", "participation", "max_drawdown")}})
FINAL = pd.DataFrame(rows).set_index("方策")
display(FINAL.round(4)); FINAL.to_csv(OUT / "final_test2025.csv")

In [ ]:
# tp-08 の方策（RL 基本）との回収率の差（paired bootstrap）
RID = ref_pk.race_ids
rng = np.random.default_rng(0)
def arrs(k):
    s = SET[k].set_index("race_id").reindex(RID)
    return s["ret"].to_numpy(), s["stake"].to_numpy()
def roi_diff(ka, kb, n_boot=2000):
    ra, sa = arrs(ka); rb, sb = arrs(kb)
    f = lambda ii: ra[ii].sum() / max(sa[ii].sum(), 1e-12) - rb[ii].sum() / max(sb[ii].sum(), 1e-12)
    bs = np.array([f(rng.integers(0, len(RID), len(RID))) for _ in range(n_boot)])
    return f(np.arange(len(RID))), np.quantile(bs, 0.025), np.quantile(bs, 0.975)
rows = []
for k in SET:
    if k == "RL 基本（tp-08）":
        continue
    d, lo, hi = roi_diff(k, "RL 基本（tp-08）")
    rows.append({"方策": k, "回収率の差(相手 − RL基本)": d, "95%下限": lo, "95%上限": hi})
DIFF = pd.DataFrame(rows).set_index("方策")
display(DIFF.round(4)); DIFF.to_csv(OUT / "final_vs_rl_base.csv")

In [ ]:
# 購入時点のオッズのずれ: (i) 雑音なしで学習した方策に雑音つきオッズを与える（頑健性）
rows = []
dtr_c, _ = prepare(TR_Y, TEST_Y)
for name, spec in RL_METHODS.items():
    pk_tr, mu, sd = pol.pack(dtr_c, groups=spec["groups"])
    ws = {}
    nets = []
    for s in FINAL_SEEDS:
        c = pol.TrainConfig(**{**BEST8, "mode": spec["mode"], "seed": s, "threads": THREADS})
        nets.append((pol.train_policy(pk_tr, c)[0], c))
    for sg in (0.0, 0.05, 0.1, 0.2):
        dte_n = prepare(TR_Y, TEST_Y, 0.0, sg)[1]
        pk_n, _, _ = pol.pack(dte_n, mu, sd, groups=spec["groups"])
        w = np.mean([pol.predict_alloc(n_, pk_n, c_.max_ticket) for n_, c_ in nets], axis=0)
        r = pol.summarize(pol.settle(pk_n, w), n_boot=500)
        rows.append({"方策": name, "判断オッズの雑音σ": sg, "学習": "雑音なし", **{k: r[k] for k in ("roi", "roi_lo", "roi_hi", "stake_per_race")}})
# (ii) 雑音つきで学習・評価
for name, spec in RL_METHODS.items():
    pk_n, w = run_rl(spec, TR_Y, TEST_Y, FINAL_SEEDS, 0.1, 0.1)
    r = pol.summarize(pol.settle(pk_n, w), n_boot=500)
    rows.append({"方策": name, "判断オッズの雑音σ": 0.1, "学習": "雑音つき(σ=0.1)", **{k: r[k] for k in ("roi", "roi_lo", "roi_hi", "stake_per_race")}})
NOISE = pd.DataFrame(rows)
display(NOISE.pivot_table(index=["方策", "学習"], columns="判断オッズの雑音σ", values="roi").round(4)); NOISE.to_csv(OUT / "final_noise.csv", index=False)
# 選別系も雑音つきオッズで
rows = []
for sg in (0.0, 0.05, 0.1, 0.2):
    dtr_s, dte_s = prepare(TR_Y, TEST_Y, sg, sg)
    pk_s, _, _ = pol.pack(dte_s)
    for name, w in select_methods(pk_s, dte_s).items():
        r = pol.summarize(pol.settle(pk_s, w), n_boot=300)
        rows.append({"方策": name, "判断オッズの雑音σ": sg, "roi": r["roi"], "roi_lo": r["roi_lo"], "roi_hi": r["roi_hi"]})
NOISE_SEL = pd.DataFrame(rows)
display(NOISE_SEL.pivot_table(index="方策", columns="判断オッズの雑音σ", values="roi").round(4)); NOISE_SEL.to_csv(OUT / "final_noise_selection.csv", index=False)

In [ ]:
res = {"evaluated_at": datetime.now().isoformat(timespec="seconds"), "tag": TAG, "git_commit": GIT_COMMIT, "test_year": TEST_Y, "smoke": SMOKE,
       "benter": {"a": A_B, "b": B_B}, "alphas": {"alpha2": AL2, "alpha3": AL3},
       "lr_test": LR.reset_index().to_dict("records"), "final": FINAL.reset_index().to_dict("records"), "vs_rl_base": DIFF.reset_index().to_dict("records"),
       "notes": ["オッズは締切後の確定オッズ（購入時点の値ではない）", "雑音の模擬は粗い代理（対数正規・買い目ごとに独立）"]}
(OUT / "report.json").write_text(json.dumps(res, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
if not SMOKE:
    with mlflow.start_run(run_name=f"tp-09-betting-compare-{TAG}", tags={"notebook": "tp-09", "git_commit": GIT_COMMIT, "test_year": "2025"}):
        mlflow.log_params({"benter_a": A_B, "benter_b": B_B, "alpha2": AL2, "alpha3": AL3, "tag": TAG})
        for k, r in FINAL.iterrows():
            mlflow.log_metrics({f"test2025_roi__{k}"[:240]: float(r["roi"])})
        for p in OUT.glob("*"):
            mlflow.log_artifact(str(p), "betting_compare")
    LOCK.write_text(json.dumps({"evaluated_at": res["evaluated_at"], "git_commit": GIT_COMMIT, "note": "test 2025 の戦略比較は実施済み。結果を見てやり直さない。"}, ensure_ascii=False))
print("完了")